In [1]:
import sys
sys.path.append("../")

In [2]:
import torch
from qmpsqsc.models import mpsqsc
from qmpsqsc.models import qmps
from importlib import reload
from qmpsqsc.models.data.utils import flip_sites_in_mps
import qmpsqsc.models.data as mpsdata

reload(mpsqsc)

<module 'qmpsqsc.models.mpsqsc' from '/Users/keisuke/Documents/projects/mps4qsc/notebooks/../qmpsqsc/models/mpsqsc/__init__.py'>

In [3]:
import torch.nn.functional as F

L = 30
chi = 2
d = 2
ghz = mpsqsc.build_ghz_state(L, d, chi).to(dtype=torch.complex128)
ghz = ghz.normalize()

zghz = ghz.copy()
zghz.As[0][:, 1] = -ghz.As[0][:, 1]

ghz_X_errors = [flip_sites_in_mps(ghz, [i], "X") for i in range(L)]
ghz_Y_errors = [flip_sites_in_mps(ghz, [i], "Y") for i in range(L)]
ghz_Z_errors = [flip_sites_in_mps(ghz, [i], "Z") for i in range(L)]

zghz_X_errors = [flip_sites_in_mps(zghz, [i], "X") for i in range(L)]
zghz_Y_errors = [flip_sites_in_mps(zghz, [i], "Y") for i in range(L)]
zghz_Z_errors = [flip_sites_in_mps(zghz, [i], "Z") for i in range(L)]

allup = mpsqsc.build_classical_state(L, d, [0]*L).to(dtype=torch.complex128)
alldown = mpsqsc.build_classical_state(L, d, [1]*L).to(dtype=torch.complex128)


In [4]:
ghzs_X = mpsqsc.add_mpstates(ghz_X_errors)
ghzs_Y = mpsqsc.add_mpstates(ghz_Y_errors)



zghzs_X = mpsqsc.add_mpstates(zghz_X_errors)
zghzs_Y = mpsqsc.add_mpstates(zghz_Y_errors)


In [5]:
from qmpsqsc.models.mpsqsc.compress import compress_mpstate

ghzs_X, ghzsX_fid = compress_mpstate(ghzs_X, 4, adam_steps=0, n_sweeps=1)
ghzs_Y, ghzsY_fid = compress_mpstate(ghzs_Y, 4, adam_steps=0, n_sweeps=1)

print("fidelities", ghzsX_fid, ghzsY_fid)

zghzs_X, zghzX_fid = compress_mpstate(zghzs_X, 4, adam_steps=0, n_sweeps=1)
zghzs_Y, zghzY_fid = compress_mpstate(zghzs_Y, 4, adam_steps=0, n_sweeps=1)

print("fidelities", zghzX_fid, zghzY_fid)


ghzs = mpsqsc.add_mpstates([ghz, ghzs_X])
ghzs = ghzs.normalize()

ghzs, ghzs_fid = compress_mpstate(ghzs, 6, adam_steps=0, n_sweeps=1)

print("fidelities", ghzs_fid)

zghzs = mpsqsc.add_mpstates([zghz, zghzs_X])
zghzs = zghzs.normalize()

zghzs, zghz_fid = compress_mpstate(zghzs, 6, adam_steps=0, n_sweeps=1)

print("fidelities", zghz_fid)


/Users/keisuke/miniconda3/envs/mpsqsc/lib/python3.11/site-packages/tenpy/networks/mps.py:1629: UserWarning: unit_cell_width is a new argument for MPS and similar classes. It is optional for now, but will become mandatory in a future release. The default value (unit_cell_width=len(sites)) is correct, iff the lattice is a Chain. For other lattices, it is incorrect. It is used for dipolar charges and correlation_function2.
  super().__init__(sites, bc, unit_cell_width)
/Users/keisuke/miniconda3/envs/mpsqsc/lib/python3.11/site-packages/tenpy/algorithms/mps_common.py:2259: UserWarning: VariationalCompression with min_sweeps=max_sweeps: we recommend to set tol_theta_diff=None to avoid overhead
  warnings.warn(
/Users/keisuke/Documents/projects/mps4qsc/notebooks/../qmpsqsc/models/mpsqsc/compress.py:36: ComplexWarning: Casting complex values to real discards the imaginary part
  return res_mps, float(psi_t.overlap(psi))


fidelities (1.0000000000000022+0j) (1.0000000000000038-6.242203018030799e-17j)
fidelities (1.000000000000004+0j) (1.0000000000000036-8.70664966950906e-16j)
fidelities (1.0000000000000002+0j)
fidelities (1.0000000000000016+0j)


In [ ]:
import torch
import torch.nn as nn
import math

class LinearSVM(nn.Module):
    def __init__(self, in_dim: int, dtype: torch.dtype = torch.float64):
        super().__init__()
        self.linear = nn.Linear(in_dim, 1, dtype=dtype)  # w^T phi(x) + b

    def forward(self, x):
        # x: (B, in_dim)
        scores = self.linear(x).squeeze(-1)  # (B,)
        return scores


def svm_hinge_loss(model: LinearSVM,
                   scores: torch.Tensor,
                   y: torch.Tensor,
                   C: float = 1.0):
    """
    scores: (B,) = f(x)
    y: (B,) with values 0 or 1
    """
    # 0/1 -> -1/+1
    y_pm1 = 2 * y.float() - 1.0

    margins = 2 - y_pm1 * scores
    hinge = torch.clamp(margins, min=0.0).mean()

    # L2 regularization on w
    w = model.linear.weight
    l2_reg = 0.002 * torch.sum(w * w) 

    return l2_reg + C * hinge


def poly2_features(V: torch.Tensor) -> torch.Tensor:
    """
    V: (B, 2) with columns [x1, x2]
    Returns phi(V): (B, 3) for degree-2 homogeneous poly kernel:
        k(x, z) = (x^T z)^2
    Feature map: [x1^2, sqrt(2)*x1*x2, x2^2]
    """
    x1 = V[:, 0:1]
    x2 = V[:, 1:1+1]

    phi = torch.cat(
        [
            x1,
            x2,
            x1 ** 2,
            math.sqrt(2.0) * x1 * x2,
            x2 ** 2,
        ],
        dim=1,
    )
    return phi

def svm_accuracy(scores, y):
    """
    scores: (B,) SVM outputs
    y: (B,) true labels in {0,1}
    """
    # Predict
    pred_pm1 = torch.sign(scores)         # -1 or +1
    pred = (pred_pm1 > 0).long()          # 0 or 1

    # Compare with labels
    correct = (pred == y).float().sum()
    acc = correct / y.numel()
    return acc

In [7]:
svm = LinearSVM(in_dim=5, dtype=torch.float64)  # because phi(V) has dim 3
svm.train()

LinearSVM(
  (linear): Linear(in_features=5, out_features=1, bias=True)
)

In [8]:
data_generator = mpsdata.ghz.create_ghz_rho_batch_qsc(ghz, allup, alldown, 2**6, 0.5, random_flip=True)

In [9]:
ghzs.set_requires_grad(True)
zghzs.set_requires_grad(True)
optimizer = torch.optim.Adam(ghzs.As + zghzs.As, lr=0.001)
optim_svm = torch.optim.Adam(svm.parameters(), lr=0.01)

optimizer.zero_grad()
optim_svm.zero_grad()
svm_train_steps = 30

for _ in range(1000):
    states, labels, errors = next(data_generator)
    # loss, acc = mpsdata.calculate_loss_mpstates(ghzs, zghzs, states, labels)
    probs, norms = mpsdata.mps_binary_predict(ghzs, zghzs, states)
    phi_V = poly2_features(probs)  # (B, 3)
    for _ in range(svm_train_steps):
        optimizer.zero_grad()
        optim_svm.zero_grad()
        scores = svm(phi_V.detach().clone())        # (B,)
        loss = svm_hinge_loss(svm, scores, labels, C=0.5)
        loss.backward()
        acc = svm_accuracy(scores, labels)
        optim_svm.step()
        # print("internal loss", loss.item(), acc.mean().item())
    optimizer.zero_grad()
    scores = svm(phi_V)        # (B,)
    loss = svm_hinge_loss(svm, scores, labels, C=0.5)
    optimizer.step()
    acc = svm_accuracy(scores, labels)
    print(loss.item(), acc.mean().item())


0.8884771965750694 0.5
0.8309596270024077 0.5
0.7167165180922996 0.5
0.630403696633335 0.859375
0.6100275850860353 0.828125
0.6188485423302978 0.78125
0.5681361941711564 0.78125
0.5023144323021582 0.796875
0.4164732225393981 0.828125
0.2580105957962925 0.890625
0.505745892379959 0.703125
0.22038206739515037 0.890625
0.33957403750674975 0.78125
0.3054556681334286 0.78125
0.27423208488353257 1.0
0.21730102830672715 1.0
0.20939312404188315 1.0
0.1801058771122725 1.0
0.17628143503657312 1.0
0.15818298189570082 1.0
0.14539826242449147 1.0
0.13887780145680104 1.0
0.13838647657584044 1.0
0.13769847959715015 1.0
0.13711349989038302 1.0
0.13699759205775594 1.0
0.13645157458449822 1.0
0.13627987990303994 1.0
0.1360076224248228 1.0
0.13573237621032921 1.0
0.1353172996824926 1.0
0.1358039056520968 1.0
0.1350843222383859 1.0
0.13499631830082934 1.0
0.1341790556579805 1.0
0.13433700638916404 1.0
0.1341064678306324 1.0
0.13382947339658918 1.0
0.13351581841374524 1.0
0.13363318387277442 1.0
0.13323868

KeyboardInterrupt: 

In [10]:
mps_qsc = mpsqsc.helper.build_qsc_from_mpstates(ghzs, zghzs)
mps_qsc = mps_qsc.truncate_bond_dimension(2)
mps_qsc.set_requires_grad(True)

In [11]:
optimizer = torch.optim.Adam(mps_qsc.As, lr=0.01)
optim_svm = torch.optim.Adam(svm.parameters(), lr=0.01)

optimizer.zero_grad()
optim_svm.zero_grad()

svm_train_steps = 30  # set number of SVM training steps per outer loop

for _ in range(1000):
    states, labels, _ = next(data_generator)
    probs, norms = mps_qsc.predict(states)
    phi_V = poly2_features(probs)  # (B, 3)
    
    # Train SVM multiple times on this minibatch (internal loop)
    for _ in range(svm_train_steps):
        optim_svm.zero_grad()
        # SVM input should not propagate gradient to mps_qsc
        scores = svm(phi_V.detach().clone())
        loss_svm = svm_hinge_loss(svm, scores, labels, C=0.5)
        loss_svm.backward()
        optim_svm.step()
    
    # Now update mps_qsc using SVM output (stop SVM gradient!)
    optimizer.zero_grad()
    scores = svm(phi_V)
    loss = svm_hinge_loss(svm, scores, labels, C=0.5)
    loss.backward()
    optimizer.step()
    optim_svm.zero_grad()
    optimizer.zero_grad()
    acc = svm_accuracy(scores, labels)
    print(loss.item(), acc.mean().item())


1.1107497071375967 0.5
0.9560362099472126 0.5625
0.879318836184184 0.5
0.7141258150258689 0.703125
0.49765704269684585 0.765625
0.3719195015288196 0.875
0.38402856206919367 0.890625
0.26069518039715345 0.953125
0.28000194714388127 0.9375
0.27632342937010534 0.9375
0.223464032448779 0.96875
0.2133596513996965 0.984375
0.23027979536637908 0.953125
0.2216804749500137 1.0
0.25202210099878697 0.96875
0.30089568741179107 0.984375
0.2134543976893426 1.0
0.17014493429863728 1.0
0.19589323024748106 0.96875
0.1789536834913535 0.984375
0.23608549482152238 0.96875
0.2497561632274109 0.96875
0.16522468079280841 1.0
0.24180268940569516 0.953125
0.16393181239134835 1.0
0.2249182754292774 0.984375
0.21148690740390041 0.984375
0.19118327867591847 1.0
0.1712038612585792 1.0
0.21303231937064956 0.984375
0.1973126633027994 1.0
0.19937024956730132 1.0
0.20173062932154084 1.0
0.14756606091759666 1.0
0.15353806628264943 1.0
0.1821244260442996 1.0
0.15589415385353128 1.0
0.15561862134842433 1.0
0.133086721790

KeyboardInterrupt: 

In [12]:
mps_qsc = mps_qsc.canonicalize(truncate=True)
Us, last = qmps.construct_unitary_from_As(mps_qsc.As)
qmps_ghz = qmps.qMPS(L, chi, d, Us=Us, last_unitary=last)
qmps_ghz.set_requires_grad(True)

In [13]:
svm_qmps = LinearSVM(in_dim=5, dtype=torch.float64)
svm_qmps.linear.weight.data[:] = svm.linear.weight.data.clone()
svm_qmps.linear.bias.data[:] = svm.linear.bias.data.clone()

In [18]:
w = 0.01
qmps_ghz.set_weights(w)

probs, norms = qmps_ghz.predict(states)
phi_V = poly2_features(probs)  # (B, 3)
scores = svm_qmps(phi_V)        # (B,)
loss = svm_hinge_loss(svm_qmps, scores, labels, C=0.5)
print(loss.item())
probs[labels==0].max(dim=-1).values.min()


0.13558842396557258


tensor(0.9917, dtype=torch.float64, grad_fn=<MinBackward1>)

In [19]:
from qmpsqsc.models.qmps.optimizer import StiefelAdam

optimizer = StiefelAdam(qmps_ghz.unitaries(), lr=0.01)
optim_svm = torch.optim.Adam(svm_qmps.parameters(), lr=0.001)
svm_train_steps = 10 # or any number you had before

for step in range(1000):
    states, labels, _ = next(data_generator)
    optimizer.zero_grad()
    optim_svm.zero_grad()
    probs, norms = qmps_ghz.predict(states)
    print(probs[labels==0].max(dim=-1).values.min())
    phi_V = poly2_features(probs)

    # SVM internal training loop (train SVM multiple times per outer step)
    for _ in range(svm_train_steps):
        scores = svm_qmps(phi_V.detach().clone())
        loss_svm = svm_hinge_loss(svm_qmps, scores, labels, C=0.5)
        loss_svm.backward()
        optim_svm.step()
        optim_svm.zero_grad()

    # Now use the SVM output in the loss, propagate grad ONLY to qmps_ghz/unitaries
    scores = svm_qmps(phi_V)
    loss = svm_hinge_loss(svm_qmps, scores, labels, C=0.5)
    loss.backward()
    optimizer.step()
    optim_svm.zero_grad()
    optimizer.zero_grad()
    acc = svm_accuracy(scores, labels)

    print(f"[step {step:5d}] loss={loss.item():.6f}  acc={acc.mean().item():.3f}")


tensor(0.9836, dtype=torch.float64, grad_fn=<MinBackward1>)
[step     0] loss=0.138989  acc=1.000
tensor(0.9567, dtype=torch.float64, grad_fn=<MinBackward1>)
[step     1] loss=0.144756  acc=1.000
tensor(0.9444, dtype=torch.float64, grad_fn=<MinBackward1>)
[step     2] loss=0.146363  acc=1.000
tensor(0.9696, dtype=torch.float64, grad_fn=<MinBackward1>)
[step     3] loss=0.143270  acc=1.000
tensor(0.9780, dtype=torch.float64, grad_fn=<MinBackward1>)
[step     4] loss=0.144021  acc=1.000
tensor(0.9448, dtype=torch.float64, grad_fn=<MinBackward1>)
[step     5] loss=0.144767  acc=1.000
tensor(0.9844, dtype=torch.float64, grad_fn=<MinBackward1>)
[step     6] loss=0.139104  acc=1.000
tensor(0.9881, dtype=torch.float64, grad_fn=<MinBackward1>)
[step     7] loss=0.137658  acc=1.000
tensor(0.9791, dtype=torch.float64, grad_fn=<MinBackward1>)
[step     8] loss=0.141204  acc=1.000
tensor(0.9450, dtype=torch.float64, grad_fn=<MinBackward1>)
[step     9] loss=0.143615  acc=1.000
tensor(0.9837, dtype

KeyboardInterrupt: 